# DP-600 Fabric Analytics Engineer Study Notebook

Welcome to the **DP-600 study companion notebook**. This notebook is organized into sections mapped directly to the core objectives of the **Implementing Analytics Solutions Using Microsoft Fabric (DP-600)** certification exam.

Use this notebook to review core concepts and run Spark/T-SQL/DAX code snippets.

## 1. OneLake SaaS Architecture

**OneLake** is a single, unified, logical data lake for your entire organization, built on top of Azure Data Lake Storage (ADLS) Gen2. Key features include:
- **Single Copy Concept**: Data is stored once in Delta Parquet format and shared between multiple engines (Spark, SQL, Power BI) without duplication.
- **Shortcuts**: Virtual pathways to external storage (S3, ADLS Gen2, GCS) or other Fabric workspaces without copying the actual files.
- **Domains**: Logical groupings of workspaces to align data ownership with business units.

## 2. Lakehouse Architecture: Files vs Tables

A Lakehouse holds two separate areas:
1. **Files**: Land zone for raw data files (CSV, JSON, XML, Parquet).
2. **Tables**: Managed structural tables stored in Delta Parquet format, automatically registered in the Fabric Metastore.

Below is the **PySpark ETL code** to read a raw CSV from the Files directory, clean it, write it to a managed Delta Table (Silver layer), and optimize it.

In [ ]:
# 1. Read raw CSV from Files (Bronze layer)
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/sales_raw.csv")

# 2. Clean and transform the data (Silver layer)
from pyspark.sql.functions import col, to_date, trim

df_silver = df_raw.select(
    col("SalesOrderNumber").alias("order_id"),
    to_date(col("OrderDate"), "yyyy-MM-dd").alias("order_date"),
    trim(col("CustomerEmail")).alias("customer_email"),
    col("UnitPrice").cast("double").alias("unit_price"),
    col("Quantity").cast("integer").alias("quantity")
).filter(col("order_id").isNotNull())

# 3. Write to Managed Tables (Silver table) with V-Order write-optimization
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_sales")

# 4. Run compaction and multi-dimensional Z-Order indexing
spark.sql("OPTIMIZE silver_sales ZORDER BY (order_date)")

## 3. Data Warehouse SQL Endpoint & Queries

Every Lakehouse has an auto-provisioned **SQL Analytics Endpoint** (read-only relational access). In contrast, a **Fabric Warehouse** is fully read-write relational, supporting standard T-SQL DDL/DML.

The code below demonstrates T-SQL performance optimization techniques like statistics creation and query formulation with partition filters (partition pruning).

In [ ]:
-- 1. Manually create statistics to optimize the query plan
CREATE STATISTICS stat_sales_date ON dbo.silver_sales(order_date);

-- 2. Relational Query demonstrating partition pruning
SELECT 
    order_date,
    COUNT(order_id) AS total_orders,
    SUM(unit_price * quantity) AS total_revenue
FROM dbo.silver_sales
WHERE order_date >= '2026-01-01' AND order_date <= '2026-06-30' -- Partition Filter
GROUP BY order_date
ORDER BY order_date DESC;

-- 3. Abstracting complexity with views (note: avoid nested views on views for performance)
CREATE VIEW dbo.v_sales_summary AS
SELECT 
    YEAR(order_date) AS order_year,
    MONTH(order_date) AS order_month,
    COUNT(order_id) AS sales_count
FROM dbo.silver_sales
GROUP BY YEAR(order_date), MONTH(order_date);

## 4. Direct Lake Mode & Fallback Analysis

**Direct Lake** is the premier connection type in Fabric. Power BI bypasses the SQL engine and loads columns directly from Delta Parquet files on OneLake into memory.

### Fallback to DirectQuery
If the capacity's memory limit is exceeded, or if Row-Level Security (RLS) is defined in the database engine, Power BI silently falls back to **DirectQuery mode**.
- **Mitigation**: Always model RLS at the **Power BI Semantic Model** level, not at the database endpoint level.

## 5. Semantic Models & DAX Optimization

Design a clean **Star Schema** with Fact and Dimension tables connected via `1:*` relationships.

Below are DAX calculations illustrating context transition, time intelligence, and optimization practices.

In [ ]:
// 1. Year-to-Date Measure (Requires dim_date marked as Date table)
Sales YTD = 
TOTALYTD(
    SUM(fact_sales[SalesAmount]),
    dim_date[Date]
)

// 2. Sales Same Period Last Year (SPLY)
Sales SPLY = 
CALCULATE(
    SUM(fact_sales[SalesAmount]),
    SAMEPERIODLASTYEAR(dim_date[Date])
)

// 3. Activating inactive relationship using USERELATIONSHIP
Shipped Sales = 
CALCULATE(
    SUM(fact_sales[SalesAmount]),
    USERELATIONSHIP(fact_sales[ShipDateKey], dim_date[DateKey])
)

// 4. Context Transition illustration inside an iterator (SUMX)
Total Transacted Product Sales = 
SUMX(
    VALUES(dim_product[ProductKey]),
    [Sales YTD] -- referencing a measure triggers context transition row-to-filter context
)

## 6. Security, Governance & Capacity Monitoring

### Workspace Roles
- **Admin & Member**: Write and manage workspace/membership.
- **Contributor**: Write and publish items. Standard developer role.
- **Viewer**: Read-only access. Cannot view Lakehouse files directly.

### Sizing & Throttling
- **Smoothing**: Background operations are averaged over a 24-hour window, interactive operations over a 10-minute window.
- **Throttling**: Progressive throttling begins if consumption exceeds capacity allocation, delaying interactive report visuals first.